In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import statsmodels.api as sm  # Make sure to import statsmodels here
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler   
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder


In [ ]:
csv=pd.read_csv("LP2_Telco-churn-last-2000 (1).csv")
ex=pd.read_excel('Telco-churn-second-2000 (1).xlsx')


In [ ]:
csv.head()

In [ ]:
csv1=pd.DataFrame(csv)
ex1=pd.DataFrame(ex)
print(csv1.describe())
print(ex1.describe())
print(csv1.isnull().sum()) 
print(csv1.head())
print(ex1.head())

In [ ]:
# customer lifetime value, CLV
csv1['MonthlyRevenue']=csv1['MonthlyCharges']
csv1['Lifetime'] = csv1['tenure'] 
csv1['CLV'] = csv1['MonthlyRevenue'] * csv1['Lifetime']
print(csv1[['customerID','CLV']].head())

In [ ]:
IQR = csv1['MonthlyCharges'].quantile(0.75) - csv1['MonthlyCharges'].quantile(0.25)
lower__MonthlyCharges_limit = csv1['MonthlyCharges'].quantile(0.25) - (IQR * 1.5)

# Calculate the upper limit for identifying outliers
upper_MonthlyCharges_limit = csv1['MonthlyCharges'].quantile(0.75) + (IQR * 1.5)

print(IQR)
print(lower__MonthlyCharges_limit )
print(upper_MonthlyCharges_limit)
MonthlyCharges_outliers = np.where(csv1['MonthlyCharges'] > upper_MonthlyCharges_limit, True,
                          np.where(csv1['MonthlyCharges'] < lower__MonthlyCharges_limit,
                                  True, False))
csv1_without_outliers = csv1.loc[~(MonthlyCharges_outliers), ]
csv1.shape, csv1_without_outliers.shape
print(upper_MonthlyCharges_limit)

In [ ]:

# Specify which columns to standardize
numeric_columns = ['MonthlyCharges']

# Create a subset of the DataFrame with only the numeric features
minmax_data = csv1[numeric_columns]

# Initialize the StandardScaler
scaler = MinMaxScaler()

# Fit and transform the selected numeric features
scaling_minmax_scaled = scaler.fit_transform(minmax_data)

print(scaling_minmax_scaled)

scaling_minmax_scaled = pd.DataFrame(scaling_minmax_scaled, columns = minmax_data.columns)

print(scaling_minmax_scaled.head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create the histogram using Seaborn
plt.figure(figsize=(10, 6))
sns.histplot(data=csv1, 
             x="MonthlyCharges", 
             hue="Churn", 
             multiple="dodge",  # Creates separate bars for each category of Churn
             kde=False,  # Optional: turn on kernel density estimate if you want a smooth line
             bins=30)  # You can adjust the number of bins

# Add title and labels
plt.title("Churn rate according to monthly charges")
plt.xlabel("Monthly Charges")
plt.ylabel("Count")

# Show the plot
plt.show()


In [ ]:

import plotly.express as px
fig=px.histogram(csv1, 
                 x="MonthlyCharges",
                 color="Churn",
                 hover_data=csv1.columns,
                 title="Churn rate according to monthly charges",
                barmode="group")
fig.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=csv1, x='MonthlyCharges', hue='Churn', bins=20, kde= False, palette='coolwarm')
plt.title('Churn rate according to monthly charges')
plt.show()

In [ ]:
IQR1 = csv1['tenure'].quantile(0.75) - csv1['tenure'].quantile(0.25)
lower__tenure_limit = csv1['tenure'].quantile(0.25) - (IQR * 1.5)
upper_tenure_limit = csv1['tenure'].quantile(0.75) + (IQR * 1.5)
print(IQR1)
print(lower__tenure_limit )
print(upper_tenure_limit)
tenure_outliers = np.where(csv1['tenure'] > upper_tenure_limit, True,
                          np.where(csv1['tenure'] < lower__MonthlyCharges_limit,
                                  True, False))
csv1_tenure_without_outliers = csv1.loc[~(tenure_outliers), ]
csv1.shape, csv1_tenure_without_outliers.shape
print(upper_tenure_limit)

In [ ]:

# Specify which columns to standardize
numeric_columns1 = ['tenure']

# Create a subset of the DataFrame with only the numeric features
minmax_data1 = csv1[numeric_columns1]

# Initialize the StandardScaler
scaler1 = MinMaxScaler()

# Fit and transform the selected numeric features
scaling_minmax_scaled1 = scaler.fit_transform(minmax_data1)

print(scaling_minmax_scaled1)

scaling_minmax_scaled1 = pd.DataFrame(scaling_minmax_scaled1, columns = minmax_data1.columns)

scaling_minmax_scaled1.head()


In [ ]:
import plotly.express as px
fig=px.histogram(csv1, 
                 x="tenure",
                 color="Churn",
                 hover_data=csv1.columns,
                 title="Churn rate according to tenure",
                 barmode="group")
fig.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=csv1, x='tenure', hue='Churn', bins=20, kde= False, palette='coolwarm')
plt.title('Churn rate according to tenure')
plt.show()

In [ ]:
# The main factors affecting customer churn is tenure and monthly charges 

In [ ]:
# we are now going to build a machile learning algorithm to predict if a customer will churn or not 
x= scaling_minmax_scaled
print(x)
# Initialize LabelEncoder
le = LabelEncoder()

le.fit(csv1['Churn'])

csv1['le_churn'] = le.transform(csv1['Churn'])

csv1.tail()
y = csv1['le_churn']

In [ ]:


# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Initialize the Linear Regression model
model = LogisticRegression(class_weight='balanced')

# Fit the model on the training data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred = model.predict(X_test) 


# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)


In [ ]:


# Initialize Random Forest with class weights
model_rf_with_weights = RandomForestClassifier(class_weight='balanced', random_state=42)

# Fit the model on the training data
model_rf_with_weights.fit(X_train, y_train)

# Make predictions on the test set
y_pred_rf_with_weights = model_rf_with_weights.predict(X_test)

# Evaluate the model with class weights
print("Evaluation with Class Weights:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf_with_weights)}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_with_weights))
print("Classification Report:")
print(classification_report(y_test, y_pred_rf_with_weights))